# Domino pip finder - training (one class: "pip")

Trains a small YOLO11 model that finds every individual pip on the table. The app's score is the
number of pips found, so there is no number to read and nothing to confuse a 3 with a 5.

Runs on **Kaggle** (free GPU), **Colab** or a local machine. Needs a GPU to be practical: with
about 100 photos a GPU run takes minutes to an hour, a CPU run takes many hours.

Run the cells in order. **Set `SMOKE_TEST = True` first** (2 epochs) to check that every cell works,
then set it to False for the real run.

## 1. Install

In [ ]:
!pip -q install ultralytics roboflow

## 2. Settings and download your labelled photos

In [ ]:
import collections
import glob
import json
import os
import random
import re
import shutil
import zipfile

import cv2
import numpy as np
import yaml

ROBOFLOW_API_KEY = 'PASTE_YOUR_KEY_HERE'
WORKSPACE, PROJECT, VERSION = 'wael-y5k8c', 'domino-pips-cr9br', 1
SMOKE_TEST = True  # True = 2 epochs, to check every cell works. Set to False for the real run.

# Photos with NO dominoes (cables, boxes, empty tables). Kaggle: add them as a Dataset and give the path of
# the folder (Kaggle unzips for you) or of the zip here. Colab: leave None, you will be asked to upload.
NEG_ZIP = None
MAX_TRAIN_NEGATIVES = 20  # your photos are few, so empties are kept to about a fifth of the training set
MAX_VAL_NEGATIVES = 10

ON_KAGGLE = os.path.exists('/kaggle/working')
ROOT = '/kaggle/working' if ON_KAGGLE else '/content' if os.path.exists('/content') else os.getcwd()
RUNS = f'{ROOT}/runs'
OUT = f'{ROOT}/merged'
HOLDOUT = f'{ROOT}/negatives_holdout'

from roboflow import Roboflow

dataset = (Roboflow(api_key=ROBOFLOW_API_KEY).workspace(WORKSPACE).project(PROJECT)
           .version(VERSION).download('yolov8', location=f'{ROOT}/raw'))
RAW = dataset.location
print(RAW)

## 3. Inspect, and check every photo's labels against its confirmed total
The scan file names contain the total you confirmed in the app (`..._total19_model29.jpg`). After
labelling, the number of pip boxes on a photo should equal that total. A difference means either a
labelling slip or a wrong confirmed total. Look at the worst ones.

In [ ]:
names = yaml.safe_load(open(f'{RAW}/data.yaml'))['names']
print('classes:', names)
assert len(names) == 1, 'expected a single class (pip)'


def parse(path):
    """(timestamp, confirmed total) from a scan file name, or (None, None)."""
    m = re.search(r'scan_(\d{8})_(\d{6})_total(\d+)_model(\d+)', os.path.basename(path))
    return (m.group(1) + m.group(2), int(m.group(3))) if m else (None, None)


photos = []
for split in ('train', 'valid', 'test'):
    for img in sorted(glob.glob(f'{RAW}/{split}/images/*')):
        label = img.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        boxes = sum(1 for l in open(label) if l.strip()) if os.path.exists(label) else 0
        stamp, total = parse(img)
        photos.append({'img': img, 'label': label, 'boxes': boxes, 'stamp': stamp or '', 'total': total})

widths = []
for p in photos:
    if os.path.exists(p['label']):
        widths += [float(l.split()[3]) for l in open(p['label']) if len(l.split()) == 5]
print(f"{len(photos)} labelled photos, {sum(p['boxes'] for p in photos)} pips, "
      f"median pip box {np.median(widths) * 100:.1f}% of the image width")

checked = [p for p in photos if p['total'] is not None]
bad = sorted((p for p in checked if p['boxes'] != p['total']), key=lambda p: -abs(p['boxes'] - p['total']))
print(f'box count equals the confirmed total on {len(checked) - len(bad)} of {len(checked)} photos; {len(bad)} differ:')
for p in bad[:40]:
    print(f"   photo {p['stamp']}: {p['boxes']} boxes vs confirmed total {p['total']}")

## 4. Split by time and add the empty photos
Neighbouring photos are near-copies, so a random split would leak them into validation and flatter the
score. Instead every fifth block of 5 consecutive photos (by capture time) becomes validation.

Set `DROP_MISMATCHED = True` to leave out the photos listed above whose counts disagree.

In [ ]:
DROP_MISMATCHED = False

usable = sorted(photos, key=lambda p: p['stamp'])
if DROP_MISMATCHED:
    usable = [p for p in usable if p['total'] is None or p['boxes'] == p['total']]
BLOCK = 5
val_idx = {i for i in range(len(usable)) if (i // BLOCK) % 5 == 2}

shutil.rmtree(OUT, ignore_errors=True)
for split in ('train', 'val'):
    os.makedirs(f'{OUT}/{split}/images')
    os.makedirs(f'{OUT}/{split}/labels')
for i, p in enumerate(usable):
    split = 'val' if i in val_idx else 'train'
    stem = f'p{i:03d}'
    shutil.copy(p['img'], f"{OUT}/{split}/images/{stem}{os.path.splitext(p['img'])[1].lower()}")
    if os.path.exists(p['label']):
        shutil.copy(p['label'], f'{OUT}/{split}/labels/{stem}.txt')
    else:
        open(f'{OUT}/{split}/labels/{stem}.txt', 'w').close()
print(f'{len(usable) - len(val_idx)} training photos, {len(val_idx)} validation photos')

# --- empty photos ---
os.makedirs(f'{ROOT}/negatives', exist_ok=True)
if NEG_ZIP and os.path.isdir(NEG_ZIP):  # Kaggle unzips uploaded zips into a folder for you
    shutil.copytree(NEG_ZIP, f'{ROOT}/negatives', dirs_exist_ok=True)
elif NEG_ZIP:
    with zipfile.ZipFile(NEG_ZIP) as z:
        z.extractall(f'{ROOT}/negatives')
else:
    from google.colab import files  # Colab only
    for name, data in files.upload().items():
        if name.lower().endswith('.zip'):
            with zipfile.ZipFile(name) as z:
                z.extractall(f'{ROOT}/negatives')
        else:
            open(f'{ROOT}/negatives/{name}', 'wb').write(data)

IMG_EXT = ('.jpg', '.jpeg', '.png', '.webp')
neg = sorted(p for p in glob.glob(f'{ROOT}/negatives/**/*', recursive=True) if p.lower().endswith(IMG_EXT))
random.Random(0).shuffle(neg)
n_val_neg = min(MAX_VAL_NEGATIVES, len(neg))
n_train_neg = min(MAX_TRAIN_NEGATIVES, len(neg) - n_val_neg)

shutil.rmtree(HOLDOUT, ignore_errors=True)
os.makedirs(HOLDOUT)
counts = {'train': 0, 'val': 0, 'holdout': 0}
for i, p in enumerate(neg):
    split = 'val' if i < n_val_neg else 'train' if i < n_val_neg + n_train_neg else 'holdout'
    im = cv2.imread(p)
    if im is None:
        print('skipping unreadable file:', os.path.basename(p))
        continue
    scale = 1440 / max(im.shape[:2])  # same size as the scan photos
    if scale < 1:
        im = cv2.resize(im, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    if split == 'holdout':
        cv2.imwrite(f'{HOLDOUT}/neg{i}.jpg', im, [cv2.IMWRITE_JPEG_QUALITY, 95])
    else:
        cv2.imwrite(f'{OUT}/{split}/images/neg{i}.jpg', im, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{OUT}/{split}/labels/neg{i}.txt', 'w').close()  # empty label file = "nothing here"
    counts[split] += 1
print('empty photos:', counts, '(the holdout is kept aside to measure false alarms)')

yaml.safe_dump({'path': OUT, 'train': 'train/images', 'val': 'val/images', 'names': ['pip']},
               open(f'{OUT}/data.yaml', 'w'))

## 5. Train
Starts from a general pretrained model. `imgsz=1280` keeps pips big enough to see; on a CPU lower it
(960) and use fewer epochs, or it will take many hours.

In [ ]:
import torch
from ultralytics import YOLO

EPOCHS = 2 if SMOKE_TEST else 150
IMGSZ = 1280
GPU = torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0) if GPU else 'NONE - this will be very slow')

YOLO('yolo11n.pt').train(
    data=f'{OUT}/data.yaml', imgsz=IMGSZ, epochs=EPOCHS, patience=50, batch=-1 if GPU else 4,
    flipud=0.5, fliplr=0.5, degrees=0, scale=0.5, mosaic=1.0, close_mosaic=15,
    project=RUNS, name='pips', exist_ok=True)

## 6. Pip-count accuracy on validation photos, plus false alarms
The predicted total is simply the number of pips found. Compared with the number of pips labelled.

In [ ]:
best = YOLO(f'{RUNS}/pips/weights/best.pt')
CONFS = (0.2, 0.3, 0.4, 0.5, 0.6, 0.7)
val_imgs = sorted(glob.glob(f'{OUT}/val/images/*'))
holdout_imgs = sorted(glob.glob(f'{HOLDOUT}/*.jpg'))


def confidences(path):
    r = best.predict(path, imgsz=IMGSZ, conf=min(CONFS), iou=0.5, max_det=500, verbose=False)[0]
    return r.boxes.conf.tolist()


def n_at(confs, threshold):
    return sum(1 for c in confs if c >= threshold)


def labelled_pips(img):
    lf = img.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    return sum(1 for l in open(lf) if l.strip())


truth = np.array([labelled_pips(p) for p in val_imgs])
val_conf = [confidences(p) for p in val_imgs]
mae = {}
for conf in CONFS:
    err = np.array([n_at(c, conf) for c in val_conf]) - truth
    mae[conf] = float(np.mean(np.abs(err)))
    print(f'conf {conf}: exact {np.mean(err == 0):.0%} | within 1 pip {np.mean(np.abs(err) <= 1):.0%} | '
          f'mean abs error {mae[conf]:.2f} | mean signed {np.mean(err):+.2f}')
BEST_CONF = min(mae, key=mae.get)
print('best confidence:', BEST_CONF)

if holdout_imgs:
    hold = [confidences(p) for p in holdout_imgs]
    alarms = sum(1 for c in hold if n_at(c, BEST_CONF) > 0)
    print(f'false alarms on {len(hold)} unseen empty photos: {alarms} ({alarms / len(hold):.0%})')

## 6b. The honest test: fresh scan photos you did NOT label or train on
Take new photos with the app (Settings -> save scan photos), confirm the right totals, and put the zip
here. Their file names carry the confirmed totals. Skip this cell until you have them.

In [ ]:
FRESH_ZIP = None  # e.g. '/kaggle/input/fresh/Counter test pics.zip'; on Colab leave None to upload
os.makedirs(f'{ROOT}/fresh', exist_ok=True)
if FRESH_ZIP and os.path.isdir(FRESH_ZIP):  # Kaggle unzips uploaded zips into a folder for you
    shutil.copytree(FRESH_ZIP, f'{ROOT}/fresh', dirs_exist_ok=True)
elif FRESH_ZIP:
    with zipfile.ZipFile(FRESH_ZIP) as z:
        z.extractall(f'{ROOT}/fresh')
else:
    from google.colab import files
    for name, data in files.upload().items():
        if name.lower().endswith('.zip'):
            with zipfile.ZipFile(name) as z:
                z.extractall(f'{ROOT}/fresh')

fresh = []
for p in sorted(glob.glob(f'{ROOT}/fresh/**/*.jpg', recursive=True)):
    stamp, total = parse(p)
    if total is not None:
        fresh.append((p, total))
assert fresh, 'no files named like scan_..._total14_model12.jpg were found'
fresh_truth = np.array([t for _, t in fresh])
fresh_conf = [confidences(p) for p, _ in fresh]
print(f'{len(fresh)} fresh photos')
for conf in CONFS:
    err = np.array([n_at(c, conf) for c in fresh_conf]) - fresh_truth
    print(f'conf {conf}: exact {np.mean(err == 0):.0%} | within 1 pip {np.mean(np.abs(err) <= 1):.0%} | '
          f'mean abs error {np.mean(np.abs(err)):.2f} | too high {np.mean(err > 0):.0%}')

## 7. Export for the app

In [ ]:
best.export(format='tflite', imgsz=IMGSZ, half=False)
exported = glob.glob(f'{RUNS}/pips/weights/**/*.tflite', recursive=True)
print(exported)
assert exported, 'no .tflite produced - re-run this cell once (the first run installs converters)'
model_path = sorted(exported, key=lambda p: ('float16' in p or 'int8' in p, len(p)))[0]

try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    import tensorflow as tf
    Interpreter = tf.lite.Interpreter

interp = Interpreter(model_path=model_path)
interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
print('MODEL :', model_path)
print('INPUT :', inp['shape'], inp['dtype'])
print('OUTPUT:', out['shape'], out['dtype'])

os.makedirs(f'{ROOT}/app_assets', exist_ok=True)
shutil.copy(model_path, f'{ROOT}/app_assets/pip_detector.tflite')
json.dump({'classValues': [1], 'conf': BEST_CONF, 'iou': 0.5}, open(f'{ROOT}/app_assets/pip_model.json', 'w'))
print('files ready in', f'{ROOT}/app_assets')
if ON_KAGGLE:
    from IPython.display import FileLink, display  # click the links to download
    for f in ('pip_detector.tflite', 'pip_model.json'):
        display(FileLink(f'app_assets/{f}'))
else:
    try:
        from google.colab import files
        for f in ('pip_detector.tflite', 'pip_model.json'):
            files.download(f'{ROOT}/app_assets/{f}')
    except ImportError:
        print('Not on Colab: copy both files out of', f'{ROOT}/app_assets')

Put both files in `app/src/main/assets/` (replacing the old ones) and rebuild the app.
Paste the cell 3 and cell 6 output and the MODEL / INPUT / OUTPUT lines back into the chat.